In [ ]:
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer

In [ ]:
MODELNAME = "FacebookAI/roberta-base"
# MODELNAME = "google-bert/bert-base-uncased"
# MODELNAME = "distilbert-base-uncased-finetuned-sst-2-english"
DIRNAME = "roberta"
AUGMENTED = False

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODELNAME)
tokenizer = AutoTokenizer.from_pretrained(MODELNAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
train_dataset = load_from_disk("dataset.hf")['train-augmented' if AUGMENTED else 'train']
eval_dataset = load_from_disk("dataset.hf")['validation']

tokenized_train_dataset = train_dataset.map(lambda ds: tokenizer(ds['text']), batched=True, remove_columns=["text"])
tokenized_eval_dataset = eval_dataset.map(lambda ds: tokenizer(ds['text']), batched=True, remove_columns=["text"])

In [ ]:
training_arguments = TrainingArguments(
    output_dir=f"models/{DIRNAME}" + ("-augmented" if AUGMENTED else ""),
    num_train_epochs=5,
    learning_rate=0.00001,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    data_collator=data_collator
)

trainer.train()

In [ ]:
tp = 0
tn = 0
fp = 0
fn = 0

guesses = [0,0]

for row in eval_dataset:
    prompt = row['text']
    label = row['label']

    with torch.no_grad():
        tokens = tokenizer(prompt, return_tensors='pt').to('cuda')
        tokenized_output = model(**tokens)
    
    guess = tokenized_output['logits'].argmax()
    guesses[guess] += 1
    if guess:
        if label:
            tp += 1
        else:
            fp += 1
    else:
        if label:
            fn += 1
        else:
            tn += 1

In [ ]:
print(f"{'Results':^20}")
print(f"Accuracy:   {(tp+tn)/len(eval_dataset):<10.2%}")
print(f"Precision:  {tp/(tp+fp):<10.2%}")
print(f"Recall:     {tp/(tp+fn):<10.2%}")
print(f"F1 Score:   {(2*tp)/(2*tp+fp+fn):<10.2%}")
print(f"Guesses:    [{guesses[0]}, {guesses[1]}]")